In [32]:
!pip install matplotlib scipy pandas cvxpy tqdm seaborn kaggle 'cvxpy[glpk]' polarix axelrod -q

/opt/homebrew/Caskroom/miniconda/base/envs/spiel311/lib/python3.11/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [33]:
import axelrod as axl                                                                                                                                                     
import numpy as np
import polarix as plx
import jax.numpy as jnp
import sys        
import pickle                                                                                                                                                         
sys.path.insert(0, "/Users/gabesmithline/Desktop/Causal-Game-Analysis")
from src.iterative_game_analysis.metagame import MetaGame
import os
import pandas as pd


In [34]:
players = [                                 
    # Cooperative                       
    axl.TitForTat(),
    axl.TitFor2Tats(),                                                                                                                                                                            
    axl.Cooperator(),
    axl.GeneralSoftGrudger(),                                                                                                                                                                     
    axl.WinStayLoseShift(),
    axl.Grudger(),
    # Exploitative
    axl.Defector(),
    axl.SuspiciousTitForTat(),
    axl.HardGoByMajority(),
    axl.Bully(),
    axl.Aggravater(),
    axl.Predator(),
    axl.BackStabber(),
    axl.DoubleCrosser(),
    # Mixed/Adaptive
    axl.Random(),
    axl.TwoTitsForTat(),
    axl.HardTitForTat(),
    axl.Prober(), 
    axl.SoftJoss(),
    axl.Prober2(),
    axl.Prober3(),
    # Sophisticated/ZD
    axl.Calculator(),
    axl.Punisher(),
    axl.InversePunisher(),
    axl.AdaptiveTitForTat(),
    axl.EvolvedFSM16(),
    axl.EvolvedFSM4(),
    axl.ThueMorse(),
    axl.Detective(),
    axl.TrickyDefector(),
]

strategy_names = [str(p) for p in players]
print(f"Strategies: {len(players)}")


Strategies: 30


In [35]:
print([s.name for s in axl.filtered_strategies({'cooperates_against_cooperator': False})])

['ALLCorALLD', 'AON2', 'Adaptive Pavlov 2006', 'Adaptive Pavlov 2011', 'Adaptive', 'Adaptive Tit For Tat', 'AdaptorBrief', 'AdaptorLong', 'Aggravater', 'Alexei', 'Alternator', 'Alternator Hunter', 'AntiCycler', 'Anti Tit For Tat', 'Appeaser', 'Arrogant QLearner', 'Average Copier', 'BackStabber', 'Better and Better', 'Bully', 'Burn Both Ends', 'Bush Mosteller', 'Calculator', 'CAPRI', 'Cautious QLearner', 'CollectiveStrategy', 'Contrite Tit For Tat', 'Cooperator', 'Cooperator Hunter', 'Cycle Hunter', 'Cycler CCCCCD', 'Cycler CCCD', 'Cycler CCCDCD', 'Cycler CCD', 'Cycler DC', 'Cycler DDC', 'DBS', 'Darwin', 'Defector', 'Defector Hunter', 'Delayed AON1', 'Desperate', 'Detective', 'DoubleCrosser', 'DoubleResurrection', 'Doubler', 'Dynamic Two Tits For Tat', 'EasyGo', 'EugineNier', 'Eventual Cycle Hunter', 'Evolved ANN', 'Evolved ANN 5', 'Evolved ANN 5 Noise 05', 'Evolved FSM 16', 'Evolved FSM 16 Noise 05', 'Evolved FSM 4', 'Evolved FSM 6', 'Evolved HMM 5', 'EvolvedLookerUp1_1_1', 'EvolvedLoo

In [36]:
strats = axl.filtered_strategies({
    'long_run_time': False,
    'max_memory_depth': 100,
})
#players = [s() for s in strats[:20]]
strategy_names = [str(p) for p in players]
n = len(players)
print(f"Strategies: {n}")
turns = 200 #length of each matchup
reptitions = 200 # number of matchups each pair plays independently 
noise = .01 # % change each action gets flipped (intended C becomes D, or vice versa). This breaks deterministic strategies and differentiates robust vs fragile cooperators

'''
axl.Game(r, s, t, p) lets you set any 2x2 symmetric game:

           C        D
  C     (r,r)    (s,t)
  D     (t,s)    (p,p)

  So you can do:

  - PD: t > r > p > s (e.g., t=5, r=3, p=1, s=0)
  - Hawk-Dove/Chicken: t > r > s > p (e.g., t=5, r=3, s=1, p=0)
  - Stag Hunt: r > t > p > s (e.g., r=5, t=3, p=1, s=0)
  - Coordination: r > t, p > s (e.g., r=5, t=0, p=3, s=0)
'''
#(C, C) cell -> payoff when you both cooperate
r = 7
#(D, D) cell -> payoff when you both defect 
p = 4
#(D, C) cell -> payoff when you defect and opponent cooperates
t = 10
#(C, D) cell -> payoff when you cooperate and opponent defects 
s=0
game = axl.Game(r=r, s=s, t=t, p=p)  # higher temptation

tournament = axl.Tournament(players, turns=turns, repetitions=reptitions, noise=noise, game=game)
results = tournament.play()

# Per-repetition payoff and cooperation data for bootstrapping
# interactions[(i,j)] = list of repetitions, each a list of (C/D, C/D) turns
n_reps = len(results.payoffs[0][0])
payoff_reps = np.zeros((n_reps, n, n))
coop_reps = np.zeros((n_reps, n, n))


for i in range(n):
    for j in range(n):
        payoff_reps[:, i, j] = results.payoffs[i][j]

# For cooperation, normalised_cooperation is only the mean
# Check if per-rep cooperation is available
print(f"cooperation shape: {np.array(results.cooperation).shape}")
print(f"cooperation[0][0]: {results.cooperation[0][0]}")

# Mean matrices
payoff_matrix = payoff_reps.mean(axis=0)
coop_matrix = np.array(results.normalised_cooperation)

# NW matrix: geometric mean of both players' scores
nw_matrix = np.sqrt(payoff_matrix * payoff_matrix.T)

print(f"Payoff matrix: [{payoff_matrix.min():.2f}, {payoff_matrix.max():.2f}]")
print(f"Cooperation matrix: [{coop_matrix.min():.2f}, {coop_matrix.max():.2f}]")
print(f"NW matrix: [{nw_matrix.min():.2f}, {nw_matrix.max():.2f}]")

# Save everything for bootstrapping
data = {
    'strategy_names': strategy_names,
    'payoff_reps': payoff_reps,      # (100, n, n) - bootstrap over axis 0
    'coop_reps': coop_reps,          # (100, n, n)
    'payoff_matrix': payoff_matrix,  # (n, n) mean
    'coop_matrix': coop_matrix,      # (n, n) mean
    'nw_matrix': nw_matrix,          # (n, n) mean
    'n_strategies': n,
    'turns': turns,
    'repetitions': reptitions,
    'noise': noise
}

os.makedirs('pd_data', exist_ok=True)
with open('pd_data/pd_tournament.pkl', 'wb') as f:
    pickle.dump(data, f)

print(f"Saved to data/pd_tournament.pkl")



Strategies: 30


Analysing: 100%|██████████| 25/25 [00:01<00:00, 14.88it/s]

cooperation shape: (30, 30)
cooperation[0][0]: 24442
Payoff matrix: [0.10, 9.92]
Cooperation matrix: [0.01, 0.99]
NW matrix: [0.99, 6.96]
Saved to data/pd_tournament.pkl


In [37]:
print(f"Strategies: {len(strategy_names)}")                                                                                                                               
print(f"Matrix shape: {payoff_matrix.shape}")

Strategies: 30
Matrix shape: (30, 30)


In [65]:
payoff_matrix2 = payoff_matrix
mg_full = MetaGame(policies=strategy_names, payoff_matrix=payoff_matrix2)
sigma_full = mg_full.solve("mene")

support = [(name, sigma_full[i]) for i, name in enumerate(strategy_names) if sigma_full[i] > 1e-3]
print(f"Equilibrium support: {len(support)} strategies")
for name, w in sorted(support, key=lambda x: -x[1]):
    print(f"  {name}: {w:.4f}")



Equilibrium support: 5 strategies
  Predator: 0.7300
  Detective: 0.1424
  Prober 2: 0.0780
  Hard Go By Majority: 0.0264
  Adaptive Tit For Tat: 0.5: 0.0231


In [39]:
# from evaluation.original_paper_analysis import _solve_maxent_cce

# sigma = _solve_maxent_cce(payoff_matrix2, strategy_names)

# support = [(name, sigma[i]) for i, name in enumerate(strategy_names) if sigma[i] > 1e-3]
# print(f"Equilibrium support: {len(support)} strategies")
# for name, w in sorted(support, key=lambda x: -x[1]):
#     print(f"  {name}: {w:.4f}")


In [40]:
df = pd.DataFrame(payoff_matrix, index=strategy_names, columns=strategy_names)
print(df.round(2).to_string())

                                   Tit For Tat  Tit For 2 Tats  Cooperator  General Soft Grudger: n=1,d=4,c=2  Win-Stay Lose-Shift  Grudger  Defector  Suspicious Tit For Tat  Hard Go By Majority  Bully  Aggravater  Predator  BackStabber: (D, D)  DoubleCrosser: (D, D)  Random: 0.5  Two Tits For Tat  Hard Tit For Tat  Prober  Soft Joss: 0.9  Prober 2  Prober 3  Calculator  Punisher  Inverse Punisher  Adaptive Tit For Tat: 0.5  Evolved FSM 16  Evolved FSM 4  ThueMorse  Detective  Tricky Defector
Tit For Tat                               5.60            6.98        6.98                               6.75                 5.60     4.68      3.96                    5.24                 5.23   5.24        3.96      4.01                 6.35                   6.65         5.25              4.77              4.64    5.52            6.54      6.91      5.18        5.05      4.75              4.77                       5.87            5.96           6.86       5.17       5.67             4.79
Tit 

In [41]:
df_coop = pd.DataFrame(coop_matrix, index=strategy_names, columns=strategy_names)
print("\nCooperation Matrix:")
print(df_coop.round(2).to_string())    


Cooperation Matrix:
                                   Tit For Tat  Tit For 2 Tats  Cooperator  General Soft Grudger: n=1,d=4,c=2  Win-Stay Lose-Shift  Grudger  Defector  Suspicious Tit For Tat  Hard Go By Majority  Bully  Aggravater  Predator  BackStabber: (D, D)  DoubleCrosser: (D, D)  Random: 0.5  Two Tits For Tat  Hard Tit For Tat  Prober  Soft Joss: 0.9  Prober 2  Prober 3  Calculator  Punisher  Inverse Punisher  Adaptive Tit For Tat: 0.5  Evolved FSM 16  Evolved FSM 4  ThueMorse  Detective  Tricky Defector
Tit For Tat                               0.61            0.98        0.98                               0.91                 0.60     0.25      0.03                    0.51                 0.46   0.50        0.03      0.05                 0.79                   0.88         0.51              0.28              0.24    0.58            0.87      0.96      0.49        0.44      0.28              0.28                       0.68            0.73           0.94       0.50       0.63 

In [66]:
#holdout analysis:
with open('pd_data/pd_tournament.pkl', 'rb') as f:
    pd_data = pickle.load(f)
strategy_names = pd_data['strategy_names']
payoff_reps = pd_data['payoff_reps']       # (100, n, n)
coop_matrix = pd_data['coop_matrix']        # (n, n) mean cooperation
n = len(strategy_names)
n_bootstrap = 100
rng = np.random.default_rng(42)

metrics = ['payoff', 'coop']
metric_labels = {'payoff': 'delta Payoff', 'nw': 'delta NW', 'coop': 'delta Coop'}

# Storage: full and LOO welfare per bootstrap
results = {m: {'full': []} for m in metrics}
for s in strategy_names:
    for m in metrics:
        results[m][f'loo_{s}'] = []

# Store sigmas for support filtering
sigma_samples = []

n_reps = payoff_reps.shape[0]
for b in range(n_bootstrap):
    
    boot_payoff = np.zeros((n, n)) #resample repititions indepdently
    for i in range(n):
        for j in range(n):
            idx = rng.choice(n_reps, size=n_reps, replace=True)
            boot_payoff[i, j] = payoff_reps[idx, i, j].mean()
    boot_payoff_sym = boot_payoff 

    #Full game equilibrium for bootstrap
    mg = MetaGame(policies=strategy_names, payoff_matrix=boot_payoff_sym)
    sigma = mg.solve("mene")
    sigma_samples.append(sigma)

    # Evaluate metrics at equilibrium
    results['payoff']['full'].append(float(sigma @ boot_payoff_sym @ sigma))
    #results['nw']['full'].append(float(sigma @ nw_matrix @ sigma))
    results['coop']['full'].append(float(sigma @ coop_matrix @ sigma))

    # LOO
    for i, s in enumerate(strategy_names):
        loo_idx = [j for j in range(n) if j != i]
        loo_payoff = boot_payoff_sym[np.ix_(loo_idx, loo_idx)]
        loo_names = [strategy_names[j] for j in loo_idx]

        mg_loo = MetaGame(policies=loo_names, payoff_matrix=loo_payoff)
        sigma_loo = mg_loo.solve("mene")

        results['payoff'][f'loo_{s}'].append(float(sigma_loo @ loo_payoff @ sigma_loo))
        #results['nw'][f'loo_{s}'].append(float(sigma_loo @ nw_matrix[np.ix_(loo_idx, loo_idx)] @ sigma_loo))
        results['coop'][f'loo_{s}'].append(float(sigma_loo @ coop_matrix[np.ix_(loo_idx, loo_idx)] @ sigma_loo))

    if (b + 1) % 100 == 0:
        print(f"  {b+1}/{n_bootstrap} done")

sigmas = np.array(sigma_samples)

# Report
header = '| **Strategy** | **n** | ' + ' | '.join(f'**{metric_labels[m]}**' for m in metrics) + ' |'
sep = '| --- | --- | ' + ' | '.join('---' for _ in metrics) + ' |'
print(header)
print(sep)

for s_idx, s in enumerate(strategy_names):
    mask = sigmas[:, s_idx] >= 10e-13
    n_active = int(mask.sum())
    if n_active < 5:
        continue

    row = f'| {s} | {n_active} |'
    for m in metrics:
        full_vals = np.array(results[m]['full'])[mask]
        loo_vals = np.array(results[m][f'loo_{s}'])[mask]
        diffs = np.round(full_vals - loo_vals, 6)

        mean = np.mean(diffs)
        lo, hi = np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

        stars = ''
        if lo > 0 or hi < 0:
            stars = '**'

        row += f' {mean:+.4f} [{lo:.4f}, {hi:.4f}]{stars} |'
    print(row)

/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.0001 (original failure: Solution has regret 0.000015 > 1e-05).
  warnings.warn(
/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.0001 (original failure: Solution has regret 0.000018 > 1e-05).
  warnings.warn(
/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.0001 (original failure: Solution has regret 0.000012 > 1e-05).
  warnings.warn(
/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.0001 (original failure: Solution has regret 0.000014 > 1e-05).
  warnings.warn(


  100/100 done
| **Strategy** | **n** | **delta Payoff** | **delta Coop** |
| --- | --- | --- | --- |
| Tit For Tat | 9 | +0.0024 [-0.0037, 0.0258] | +0.0008 [-0.0013, 0.0086] |
| Tit For 2 Tats | 24 | -0.1583 [-2.1499, 0.1651] | -0.0629 [-0.7171, 0.0430] |
| Hard Go By Majority | 72 | -1.1379 [-2.3515, 0.4906] | -0.3752 [-0.7727, 0.1852] |
| Predator | 79 | -2.0232 [-2.3513, 0.5525] | -0.6686 [-0.7727, 0.1905] |
| BackStabber: (D, D) | 17 | +0.9898 [-0.0337, 2.3044] | +0.3274 [-0.0141, 0.7617] |
| DoubleCrosser: (D, D) | 26 | +1.0425 [-2.1631, 2.3364] | +0.3462 [-0.7385, 0.7633] |
| Soft Joss: 0.9 | 11 | -0.2973 [-1.8268, 0.0811] | -0.1027 [-0.6294, 0.0266] |
| Prober 2 | 71 | -0.0313 [-0.0772, 0.0171] | -0.0119 [-0.0294, 0.0062] |
| Adaptive Tit For Tat: 0.5 | 70 | +0.0873 [-0.0555, 2.2372] | +0.0290 [-0.0190, 0.7519] |
| Evolved FSM 16 | 16 | +1.8806 [-0.0429, 2.6430] | +0.6267 [-0.0141, 0.8785] |
| Detective | 73 | -1.2386 [-2.3515, -0.0169]** | -0.4088 [-0.7727, -0.0034]** |


In [ ]:
"""
/Users/gabesmithline/Desktop/Causal-Game-Analysis/src/iterative_game_analysis/solvers/mene.py:73: UserWarning: MENE solution accepted with relaxed tolerance 0.0001 (original failure: Solution has regret 0.000019 > 1e-05).
  warnings.warn(
  100/100 done
| **Strategy** | **n** | **delta Payoff** | **delta Coop** |
| --- | --- | --- | --- |
| Cooperator | 100 | +0.0396 [0.0073, 0.0800]** | +0.0156 [0.0081, 0.0178]** |
| Win-Stay Lose-Shift | 99 | -0.3956 [-0.4253, -0.3522]** | -0.1059 [-0.1124, -0.1027]** |
| Soft Joss: 0.9 | 95 | +0.0017 [-0.2562, 0.0427] | -0.0066 [-0.0721, -0.0003]** |
"""

In [ ]:
#interaction effects
from itertools import combinations

n_bootstrap = 100
n_reps = payoff_reps.shape[0]
rng = np.random.default_rng(42)

test_strategies = strategy_names  # or pick a subset


interaction_results = {}

for b in range(n_bootstrap):
    boot_payoff = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            idx = rng.choice(n_reps, size=n_reps, replace=True)
            boot_payoff[i, j] = payoff_reps[idx, i, j].mean()
    boot_payoff_sym = boot_payoff 

    # Full game
    mg = MetaGame(policies=strategy_names, payoff_matrix=boot_payoff_sym)
    sigma = mg.solve("mene")
    W_full = float(sigma @ boot_payoff_sym @ sigma)
    C_full = float(sigma @ coop_matrix @ sigma)

    # Cache LOO results
    loo_cache = {}
    for i, s in enumerate(strategy_names):
        loo_idx = [j for j in range(n) if j != i]
        loo_payoff = boot_payoff_sym[np.ix_(loo_idx, loo_idx)]
        loo_names = [strategy_names[j] for j in loo_idx]
        mg_loo = MetaGame(policies=loo_names, payoff_matrix=loo_payoff)
        sigma_loo = mg_loo.solve("mene")
        loo_cache[s] = {
            'payoff': float(sigma_loo @ loo_payoff @ sigma_loo),
            'coop': float(sigma_loo @ coop_matrix[np.ix_(loo_idx, loo_idx)] @ sigma_loo),
        }

    # LTO pairs
    for s_a, s_b in combinations(strategy_names, 2):
        i_a = strategy_names.index(s_a)
        i_b = strategy_names.index(s_b)
        lto_idx = [j for j in range(n) if j != i_a and j != i_b]
        lto_payoff = boot_payoff_sym[np.ix_(lto_idx, lto_idx)]
        lto_names = [strategy_names[j] for j in lto_idx]
        mg_lto = MetaGame(policies=lto_names, payoff_matrix=lto_payoff)
        sigma_lto = mg_lto.solve("mene")
        W_lto = float(sigma_lto @ lto_payoff @ sigma_lto)
        C_lto = float(sigma_lto @ coop_matrix[np.ix_(lto_idx, lto_idx)] @ sigma_lto)

        key = (s_a, s_b)
        if key not in interaction_results:
            interaction_results[key] = {'payoff': [], 'coop': []}

        # Harsanyi dividend
        for m, W_f, W_a, W_b, W_ab in [
            ('payoff', W_full, loo_cache[s_a]['payoff'], loo_cache[s_b]['payoff'], W_lto),
            ('coop', C_full, loo_cache[s_a]['coop'], loo_cache[s_b]['coop'], C_lto),
        ]:
            dividend = W_f - W_a - W_b + W_ab
            interaction_results[key][m].append(dividend)

    if (b + 1) % 10 == 0:
        print(f"  {b+1}/{n_bootstrap} done")

# Report
print(f"\n{'Pair':<55} {'Δ² Payoff':>12} {'95% CI':>24} {'Δ² Coop':>12} {'95% CI':>24}")
print("-" * 130)

for (s_a, s_b), res in sorted(interaction_results.items(), key=lambda x: -abs(np.mean(x[1]['payoff']))):
    for m in ['payoff', 'coop']:
        vals = np.array(res[m])

    p_vals = np.array(res['payoff'])
    c_vals = np.array(res['coop'])

    p_mean = np.mean(p_vals)
    p_lo, p_hi = np.percentile(p_vals, 2.5), np.percentile(p_vals, 97.5)
    p_sig = '**' if p_lo > 0 or p_hi < 0 else ''

    c_mean = np.mean(c_vals)
    c_lo, c_hi = np.percentile(c_vals, 2.5), np.percentile(c_vals, 97.5)
    c_sig = '**' if c_lo > 0 or c_hi < 0 else ''

    # if abs(p_mean) > 0.001 or abs(c_mean) > 0.001:  # filter tiny effects
    print(f"  {s_a + ' × ' + s_b:<55} {p_mean:>+.4f} [{p_lo:>+.4f}, {p_hi:>+.4f}]{p_sig:>3}   {c_mean:>+.4f} [{c_lo:>+.4f}, {c_hi:>+.4f}]{c_sig:>3}")

  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done

Pair                                                       Δ² Payoff                   95% CI      Δ² Coop                   95% CI
----------------------------------------------------------------------------------------------------------------------------------
  Win-Stay Lose-Shift × ThueMorse                         -0.4040 [-1.6930, +0.0000]      -0.1180 [-0.4917, +0.0000]   
  Tit For 2 Tats × Win-Stay Lose-Shift                    -0.2574 [-0.3940, -0.2229] **   -0.0691 [-0.1039, -0.0621] **
  Tit For 2 Tats × Cooperator                             +0.1643 [+0.0293, +0.2172] **   +0.0464 [+0.0054, +0.0578] **
  Cooperator × Soft Joss: 0.9                             -0.1546 [-0.3847, -0.1078] **   -0.0560 [-0.1090, -0.0479] **
  Tit For 2 Tats × Evolved FSM 4                          -0.0243 [-0.1535, +0.0000]      -0.0070 [-0.0405, +0.0000

In [ ]:
"""
10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done

Pair                                                       Δ² Payoff                   95% CI      Δ² Coop                   95% CI
----------------------------------------------------------------------------------------------------------------------------------
  Win-Stay Lose-Shift × ThueMorse                         -0.4040 [-1.6930, +0.0000]      -0.1180 [-0.4917, +0.0000]   
  Tit For 2 Tats × Win-Stay Lose-Shift                    -0.2574 [-0.3940, -0.2229] **   -0.0691 [-0.1039, -0.0621] **
  Tit For 2 Tats × Cooperator                             +0.1643 [+0.0293, +0.2172] **   +0.0464 [+0.0054, +0.0578] **
  Cooperator × Soft Joss: 0.9                             -0.1546 [-0.3847, -0.1078] **   -0.0560 [-0.1090, -0.0479] **
  Tit For 2 Tats × Evolved FSM 4                          -0.0243 [-0.1535, +0.0000]      -0.0070 [-0.0405, +0.0000]   
  Cooperator × Win-Stay Lose-Shift                        +0.0219 [-0.1280, +0.0812]      +0.0087 [-0.0505, +0.0180]   
  Tit For 2 Tats × Prober 2                               -0.0217 [-0.1535, +0.0000]      -0.0064 [-0.0405, +0.0000]   
  Soft Joss: 0.9 × Adaptive Tit For Tat: 0.5              -0.0052 [-0.0822, +0.0000]      -0.0026 [-0.0437, +0.0000]   
  Tit For 2 Tats × Adaptive Tit For Tat: 0.5              -0.0050 [-0.0783, +0.0000]      -0.0012 [-0.0200, +0.0008]   
  Tit For 2 Tats × Soft Joss: 0.9                         -0.0020 [-0.2703, +0.0976]      -0.0124 [-0.0771, -0.0013] **
  Win-Stay Lose-Shift × Soft Joss: 0.9                    +0.0016 [-0.2072, +0.0425]      -0.0063 [-0.0602, -0.0000] **
  Tit For 2 Tats × DoubleCrosser: (D, D)                  -0.0013 [-0.0000, +0.0000]      -0.0002 [-0.0000, +0.0000]   
  Soft Joss: 0.9 × Evolved FSM 4                          +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Bully × Soft Joss: 0.9                                  +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober × Soft Joss: 0.9                                 +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Soft Joss: 0.9 × Punisher                               +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Predator × Soft Joss: 0.9                               +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Calculator                        +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Bully                             +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Soft Joss: 0.9                       +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Punisher                          +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Hard Tit For Tat                  +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Detective                         +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Defector                          +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Soft Joss: 0.9 × Inverse Punisher                       +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Defector × Soft Joss: 0.9                               +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Soft Joss: 0.9 × ThueMorse                              +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Soft Joss: 0.9 × Detective                              +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Soft Joss: 0.9                  +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Aggravater × Soft Joss: 0.9                             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Grudger × Soft Joss: 0.9                                -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Soft Joss: 0.9 × Prober 2                               +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Random: 0.5                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Predator                                  +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Bully                                     +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Cooperator                                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Bully × Prober 2                                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Prober              -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Predator × Calculator                                   +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × ThueMorse                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Prober                                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Evolved FSM 16                            -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Cooperator × BackStabber: (D, D)                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Tricky Defector                           +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Prober 2            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × ThueMorse                                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Aggravater                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Aggravater                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Prober 2 × Adaptive Tit For Tat: 0.5                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Defector × Calculator                                   +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Punisher                                  +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Prober 2 × Inverse Punisher                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Punisher × Evolved FSM 16                               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × DoubleCrosser: (D, D)             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Prober 3                        -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Adaptive Tit For Tat: 0.5                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Punisher × Adaptive Tit For Tat: 0.5                    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × Random: 0.5                                -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Cooperator × Adaptive Tit For Tat: 0.5                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Adaptive Tit For Tat: 0.5         +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Detective                                 -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Suspicious Tit For Tat            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Adaptive Tit For Tat: 0.5                 -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × ThueMorse                                 -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Inverse Punisher × Adaptive Tit For Tat: 0.5            -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Random: 0.5                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Calculator × Inverse Punisher                           -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Evolved FSM 4 × Detective                               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Predator            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Soft Joss: 0.9 × Tricky Defector                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Two Tits For Tat                          +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Hard Tit For Tat                          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Calculator                                -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Adaptive Tit For Tat: 0.5 × Evolved FSM 16              -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Cooperator × Prober 2                                   +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Adaptive Tit For Tat: 0.5         +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Soft Joss: 0.9                 -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Prober                            -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Predator × Inverse Punisher                             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Predator × Evolved FSM 4                                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Suspicious Tit For Tat                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Punisher                                  +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Aggravater                                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Prober 2                                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Cooperator × Predator                                   +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Evolved FSM 16                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Calculator                                    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Predator × DoubleCrosser: (D, D)                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Inverse Punisher                -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × ThueMorse                      -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Aggravater × Tricky Defector                            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × ThueMorse                         -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × DoubleCrosser: (D, D)                         -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Random: 0.5                                   -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Defector × BackStabber: (D, D)                          +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Bully × BackStabber: (D, D)                             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Aggravater × Random: 0.5                                -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Hard Tit For Tat                -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Defector                                  +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Punisher                                     +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Prober 2                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Evolved FSM 16 × Tricky Defector                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Two Tits For Tat               +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Soft Joss: 0.9                    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Calculator                        -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Bully × Tricky Defector                                 +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Inverse Punisher × Evolved FSM 4                        -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × ThueMorse                                 +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Predator                          +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Predator × Evolved FSM 16                               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Detective                         +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Soft Joss: 0.9                            -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Inverse Punisher                          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober 2 × Punisher                                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × BackStabber: (D, D) +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Prober 3            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Tricky Defector                   +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Evolved FSM 16                 -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Two Tits For Tat                  +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × BackStabber: (D, D)                    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × Prober 3                                   -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Cooperator × Punisher                                   -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Evolved FSM 4                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Suspicious Tit For Tat                 -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Tricky Defector     +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Bully × Punisher                                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Punisher                        -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober 3 × Inverse Punisher                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Defector                                      +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × DoubleCrosser: (D, D)             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Evolved FSM 4                             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Random: 0.5                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Aggravater × Hard Tit For Tat                           -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Predator × Tricky Defector                              +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Punisher × Inverse Punisher                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Tricky Defector                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Aggravater                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Aggravater × BackStabber: (D, D)                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Prober 3                                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Evolved FSM 4                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Soft Joss: 0.9                            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Prober 3                                  +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Grudger × BackStabber: (D, D)                           +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Hard Go By Majority                          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Calculator × ThueMorse                                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Tit For 2 Tats                            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Random: 0.5                            -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Evolved FSM 4       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Suspicious Tit For Tat                        -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober 2 × Detective                                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Evolved FSM 16 × Evolved FSM 4                          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Soft Joss: 0.9      +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Evolved FSM 4                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Bully × Random: 0.5                                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × Prober 2                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Calculator × Evolved FSM 16                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Punisher × ThueMorse                                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Adaptive Tit For Tat: 0.5                 +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Prober 3                                     +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Prober                          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Tricky Defector                      +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Soft Joss: 0.9 × Calculator                             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × DoubleCrosser: (D, D) -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Hard Go By Majority               +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Bully × Predator                                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Predator × Random: 0.5                                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober 2 × Calculator                                   -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Hard Tit For Tat                          +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × Defector                                   +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Hard Go By Majority -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × DoubleCrosser: (D, D)          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober × Calculator                                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Two Tits For Tat                              -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Defector × Predator                                     +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Prober 3                          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Predator × Punisher                                     +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × Inverse Punisher                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober × Prober 2                                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Calculator          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Bully × Aggravater                                      +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Hard Go By Majority                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Bully                                         +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Grudger × Prober 3                                      +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Inverse Punisher               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Prober 2                          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Bully × Two Tits For Tat                                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Tricky Defector                 -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober × Evolved FSM 4                                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Cooperator × General Soft Grudger: n=1,d=4,c=2          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Cooperator × Inverse Punisher                           +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Grudger × ThueMorse                                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × DoubleCrosser: (D, D)             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Bully × DoubleCrosser: (D, D)                           -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × Prober                               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × Soft Joss: 0.9                       +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Soft Joss: 0.9 × Evolved FSM 16                         +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Calculator × Tricky Defector                            -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Inverse Punisher × Evolved FSM 16                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Evolved FSM 16 × ThueMorse                              -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  ThueMorse × Tricky Defector                             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Predator                               +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Bully                                        -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Predator × ThueMorse                                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Two Tits For Tat                          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Evolved FSM 16                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober 3 × ThueMorse                                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Calculator                                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × Prober                                     +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Bully × Inverse Punisher                                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Bully × Evolved FSM 4                                   +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Aggravater × DoubleCrosser: (D, D)                      -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Predator × Adaptive Tit For Tat: 0.5                    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Prober 2                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Punisher × Detective                                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Adaptive Tit For Tat: 0.5 × Evolved FSM 4               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Cooperator × Two Tits For Tat                           -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Defector × DoubleCrosser: (D, D)                        -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Hard Tit For Tat               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Prober 3                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Predator                          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × BackStabber: (D, D)               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  ThueMorse × Detective                                   -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Random: 0.5         -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Prober 2                                      -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Defector × Suspicious Tit For Tat                       +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Adaptive Tit For Tat: 0.5         +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Tricky Defector                   -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Bully × Evolved FSM 16                                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Predator × BackStabber: (D, D)                          +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Prober                            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Prober × Prober 3                                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober × Adaptive Tit For Tat: 0.5                      -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Two Tits For Tat                       +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × DoubleCrosser: (D, D)                      -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Evolved FSM 4 × ThueMorse                               -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × Suspicious Tit For Tat                     +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × Hard Tit For Tat                           +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Aggravater          -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Random: 0.5                                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Hard Go By Majority            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Punisher × Evolved FSM 4                                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Adaptive Tit For Tat: 0.5 × Detective                   +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Evolved FSM 16 × Detective                              -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Cooperator × Calculator                                 +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × Evolved FSM 4                              -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Hard Go By Majority                           -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Hard Tit For Tat                              -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Inverse Punisher                              -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober × Tricky Defector                                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Detective           +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Bully × Adaptive Tit For Tat: 0.5                       +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Bully × ThueMorse                                       +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Prober                               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × Tricky Defector                      +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Prober × Inverse Punisher                               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober 2 × ThueMorse                                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober 3 × Evolved FSM 4                                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × Aggravater                                 +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Inverse Punisher                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Defector × Evolved FSM 16                               +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Detective                      -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Aggravater × Prober                                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Aggravater × Prober 2                                   -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Evolved FSM 4                   -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Inverse Punisher                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × Evolved FSM 16                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × Detective                            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Prober 3 × Evolved FSM 16                               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Calculator × Evolved FSM 4                              -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Grudger × Detective                                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Bully × Hard Tit For Tat                                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Two Tits For Tat                -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Adaptive Tit For Tat: 0.5 × Tricky Defector             -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Detective × Tricky Defector                             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × Grudger                                    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × Tricky Defector                            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Punisher                       +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Aggravater × Predator                                   +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Calculator                      -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Inverse Punisher × ThueMorse                            -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Prober 2                                  +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Inverse Punisher                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Evolved FSM 16                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Punisher                          +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Random: 0.5 × Tricky Defector                           -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober × Detective                                      +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Calculator × Detective                                  -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × General Soft Grudger: n=1,d=4,c=2      +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Aggravater                             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × ThueMorse                              -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Bully               +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Prober 2                                     +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Tricky Defector                              +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Aggravater × Evolved FSM 16                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Calculator                           -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Prober 3                               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Grudger             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Suspicious Tit For Tat +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Aggravater                                   -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Calculator                     -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Aggravater × Punisher                                   +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Random: 0.5                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Inverse Punisher                  +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Evolved FSM 4                     +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Detective                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Adaptive Tit For Tat: 0.5            -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × Adaptive Tit For Tat: 0.5            -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober × ThueMorse                                      -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober 3 × Adaptive Tit For Tat: 0.5                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Inverse Punisher                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Random: 0.5                       +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Two Tits For Tat                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Soft Joss: 0.9                    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Hard Tit For Tat                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober 3 × Tricky Defector                              +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Calculator                             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Hard Tit For Tat    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Two Tits For Tat                             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Detective                                    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Prober                         +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Bully × Calculator                                      +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Bully × Detective                                       +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Prober 2 × Tricky Defector                              +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Calculator × Punisher                                   -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Detective                                 +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Defector                               +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × Evolved FSM 16                             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Inverse Punisher    +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × ThueMorse           +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Bully                          +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Hard Tit For Tat                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Bully × Prober                                          +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Aggravater × Calculator                                 -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Aggravater × Inverse Punisher                           -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Aggravater × Evolved FSM 4                              -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × Prober 3                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Grudger                           -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Evolved FSM 16                    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Grudger × Evolved FSM 16                                -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × ThueMorse                         +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Detective                            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Prober × Evolved FSM 16                                 -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Bully                                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Cooperator × Bully                                      -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Cooperator × Detective                                  +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Grudger × Prober                                        -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Calculator                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Evolved FSM 4 × Tricky Defector                         +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × BackStabber: (D, D)                       +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Prober                            -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Inverse Punisher                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Defector × Prober                                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Tricky Defector                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Bully                             +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Evolved FSM 4                        +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Adaptive Tit For Tat: 0.5 +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Prober 2                          +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Aggravater × Detective                                  -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Prober 3                          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Evolved FSM 16                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober 2 × Prober 3                                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Inverse Punisher × Detective                            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Hard Tit For Tat                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Prober                                 -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × BackStabber: (D, D)               -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Predator                       +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Prober 3                             +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × Evolved FSM 4                        -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Prober 3 × Calculator                                   -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Cooperator × Hard Go By Majority                        +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Punisher            -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Tricky Defector                               -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Hard Tit For Tat                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Defector × ThueMorse                                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × ThueMorse                            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Evolved FSM 16                            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Punisher                               +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Evolved FSM 16                         -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Two Tits For Tat                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Predator × Hard Tit For Tat                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Predator × Prober 2                                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Predator × Detective                                    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Prober 2                          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × Punisher                             -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober × Punisher                                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Grudger                                +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Prober 3                          -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Grudger × Punisher                                      -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Aggravater × Prober 3                                   +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Predator × Prober 3                                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Prober 2 × Evolved FSM 16                               +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Calculator × Adaptive Tit For Tat: 0.5                  -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × General Soft Grudger: n=1,d=4,c=2         -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For 2 Tats × Detective                              -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Win-Stay Lose-Shift × Evolved FSM 4                     +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Grudger × Predator                                      -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Detective                         -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Aggravater × ThueMorse                                  -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Predator × Prober                                       +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Two Tits For Tat × ThueMorse                            -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Prober 3 × Detective                                    -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Win-Stay Lose-Shift                       -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Random: 0.5                               +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Prober                                    +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Evolved FSM 16      -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Grudger × Aggravater                                    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Bully × Prober 3                                        +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Aggravater × Two Tits For Tat                           +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Tricky Defector                   -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Prober 2                        -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Grudger                                   +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Hard Go By Majority                       -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Tit For Tat × DoubleCrosser: (D, D)                     +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Win-Stay Lose-Shift -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Two Tits For Tat    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × Adaptive Tit For Tat: 0.5      +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Aggravater × Adaptive Tit For Tat: 0.5                  +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  BackStabber: (D, D) × Hard Tit For Tat                  -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Punisher × Tricky Defector                              -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Adaptive Tit For Tat: 0.5 × ThueMorse                   -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Tit For Tat × Inverse Punisher                          +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Grudger × Evolved FSM 4                                 -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Suspicious Tit For Tat × BackStabber: (D, D)            -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Predator × Two Tits For Tat                             +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × Calculator                           +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Soft Joss: 0.9 × Prober 3                               -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Prober 2 × Evolved FSM 4                                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Prober 3 × Punisher                                     -0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Inverse Punisher × Tricky Defector                      -0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  General Soft Grudger: n=1,d=4,c=2 × Defector            +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Adaptive Tit For Tat: 0.5                    +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Defector × Evolved FSM 4                                +0.0000 [-0.0000, +0.0000]      +0.0000 [-0.0000, +0.0000]   
  Hard Go By Majority × Punisher                          +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  DoubleCrosser: (D, D) × Adaptive Tit For Tat: 0.5       +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]   
  Hard Tit For Tat × Punisher                             +0.0000 [-0.0000, +0.0000]      -0.0000 [-0.0000, +0.0000]
"""

In [ ]:
import pickle
import numpy as np
import sys
sys.path.insert(0, '..')
from src.iterative_game_analysis.metagame import MetaGame
from evaluation.curb_analysis import (
    find_minimal_curb_sets_via_closure,
    find_all_curb_sets_via_closure,
    curb_closure,
    compute_cbr,
    is_curb,
)

MC_Budget = 100 #MC budget 
bootstraps = 100 # number of bootstrapped empirical games 
rng = np.random.default_rng(42) #random seed 
with open('pd_data/pd_tournament.pkl', 'rb') as f:
    pd_data = pickle.load(f)

strategy_names = pd_data['strategy_names']
payoff_reps = pd_data['payoff_reps']
coop_matrix = pd_data['coop_matrix']
n = len(strategy_names)
n_bootstrap = 100
rng = np.random.default_rng(42)
n_reps = payoff_reps.shape[0]
metrics = ['payoff', 'coop']



#storage of curb results 
results = {m: {'full': []} for m in metrics}
for s in strategy_names:
    for m in metrics:
        results[m][f'loo_{s}'] = []

sigma_samples = []
curb_welfare_intervals = {m: {'min': [], 'max': []} for m in metrics}
curb_loo_intervals = {s: {m: {'min': [], 'max': []} for m in metrics} for s in strategy_names}
curb_set_counts = []
curb_survival = {}
curb_sets_per_bootstrap = []

for bootstrap in range(n_bootstrap):
    #get bootstrap payoff matrix
    boot_payoff = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            idx = rng.choice(n_reps, size=n_reps, replace=True)
            boot_payoff[i, j] = payoff_reps[idx, i, j].mean()
    boot_payoff_sym = boot_payoff 

    #get full meta-game 
    mg = MetaGame(policies=strategy_names, payoff_matrix=boot_payoff_sym)
    sigma = mg.solve("mene")
    sigma_samples.append(sigma)
    results['payoff']['full'].append(float(sigma @ boot_payoff_sym @ sigma))
    results['coop']['full'].append(float(sigma @ coop_matrix @ sigma))

    for i, s in enumerate(strategy_names):
        loo_idx = [j for j in range(n) if j != i]
        loo_payoff = boot_payoff_sym[np.ix_(loo_idx, loo_idx)]
        mg_loo = MetaGame(policies=[strategy_names[j] for j in loo_idx], payoff_matrix=loo_payoff)
        sigma_loo = mg_loo.solve("mene")
        results['payoff'][f'loo_{s}'].append(float(sigma_loo @ loo_payoff @ sigma_loo))
        results['coop'][f'loo_{s}'].append(float(sigma_loo @ coop_matrix[np.ix_(loo_idx, loo_idx)] @ sigma_loo))

    #curb set analysis
    #if we have <= 20 strategies 
    if n <= 20:
        all_curbs = find_all_curb_sets_via_closure(boot_payoff_sym, n)
    else: #
        print(f"Greater than 20 going to curb")
        
        minimals = find_minimal_curb_sets_via_closure(boot_payoff_sym, n)
        all_curbs = list(set(minimals))
        for _ in range(MC_Budget):
            seed_size = rng.integers(1, n + 1)
            seed = set(rng.choice(n, size=seed_size, replace=False).tolist())
            c = curb_closure(boot_payoff_sym, seed)
            if c not in all_curbs:
                all_curbs.append(c)
    
    curb_set_counts.append(len(all_curbs))
    for c in all_curbs:
        cs = frozenset(c)
        curb_survival[cs] = curb_survival.get(cs, 0) + 1

    # Welfare within each CURB set
    curb_welfares = {m: [] for m in metrics}
    curb_data = {}
    for c in all_curbs:
        c_idx = sorted(c)
        c_payoff = boot_payoff_sym[np.ix_(c_idx, c_idx)]
        mg_c = MetaGame(policies=[strategy_names[j] for j in c_idx], payoff_matrix=c_payoff)
        sigma_c = mg_c.solve("mene")
        w_p = float(sigma_c @ c_payoff @ sigma_c)
        w_c = float(sigma_c @ coop_matrix[np.ix_(c_idx, c_idx)] @ sigma_c)
        curb_welfares['payoff'].append(w_p)
        curb_welfares['coop'].append(w_c)
        curb_data[frozenset(c)] = {'idx': c_idx, 'payoff': w_p, 'coop': w_c}

    for m in metrics:
        curb_welfare_intervals[m]['min'].append(min(curb_welfares[m]))
        curb_welfare_intervals[m]['max'].append(max(curb_welfares[m]))

    # CURB-conditional LOO
    for i, s in enumerate(strategy_names):
        loo_deltas = {m: [] for m in metrics}
        for cs, cd in curb_data.items():
            if i not in cs:
                continue
            c_idx = cd['idx']
            loo_c_idx = [j for j in c_idx if j != i]
            if len(loo_c_idx) < 1:
                continue
            loo_c_payoff = boot_payoff_sym[np.ix_(loo_c_idx, loo_c_idx)]
            mg_loo_c = MetaGame(policies=[strategy_names[j] for j in loo_c_idx], payoff_matrix=loo_c_payoff)
            sigma_loo_c = mg_loo_c.solve("mene")
            loo_deltas['payoff'].append(cd['payoff'] - float(sigma_loo_c @ loo_c_payoff @ sigma_loo_c))
            loo_deltas['coop'].append(cd['coop'] - float(sigma_loo_c @ coop_matrix[np.ix_(loo_c_idx, loo_c_idx)] @ sigma_loo_c))

        for m in metrics:
            if loo_deltas[m]:
                curb_loo_intervals[s][m]['min'].append(min(loo_deltas[m]))
                curb_loo_intervals[s][m]['max'].append(max(loo_deltas[m]))
            else:
                curb_loo_intervals[s][m]['min'].append(0.0)
                curb_loo_intervals[s][m]['max'].append(0.0)

    if (bootstrap + 1) % 10 == 0:
        print(f"  {b+1}/{n_bootstrap} done")

sigmas = np.array(sigma_samples)

# ── Reports ──
print("\n=== Standard LOO ===")
header = '| **Strategy** | **n** | **delta Payoff** | **delta Coop** |'
print(header)
print('| --- | --- | --- | --- |')
for s_idx, s in enumerate(strategy_names):
    mask = sigmas[:, s_idx] >= 1e-12
    n_active = int(mask.sum())
    if n_active < 5:
        continue
    row = f'| {s} | {n_active} |'
    for m in metrics:
        full_vals = np.array(results[m]['full'])[mask]
        loo_vals = np.array(results[m][f'loo_{s}'])[mask]
        diffs = np.round(full_vals - loo_vals, 6)
        mean, lo, hi = np.mean(diffs), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)
        stars = '**' if lo > 0 or hi < 0 else ''
        row += f' {mean:+.4f} [{lo:.4f}, {hi:.4f}]{stars} |'
    print(row)

print("\n=== CURB Welfare Intervals ===")
for m in metrics:
    mn = np.mean(curb_welfare_intervals[m]['min'])
    mx = np.mean(curb_welfare_intervals[m]['max'])
    lo_ci = np.percentile(curb_welfare_intervals[m]['min'], [2.5, 97.5])
    hi_ci = np.percentile(curb_welfare_intervals[m]['max'], [2.5, 97.5])
    print(f"  {m}: [{mn:.4f}, {mx:.4f}]")
    print(f"    Lower 95% CI: [{lo_ci[0]:.4f}, {lo_ci[1]:.4f}]")
    print(f"    Upper 95% CI: [{hi_ci[0]:.4f}, {hi_ci[1]:.4f}]")
print(f"  Avg CURB sets per bootstrap: {np.mean(curb_set_counts):.1f}")

print("\n=== CURB Survival (top 10) ===")
for c, count in sorted(curb_survival.items(), key=lambda x: -x[1])[:10]:
    names = [strategy_names[i] for i in sorted(c)]
    print(f"  {count/n_bootstrap*100:5.1f}% | {{{', '.join(names)}}}")

print("\n=== CURB-Conditional LOO Intervals ===")
print(f"{'Strategy':<30} {'Min Delta':>12} {'Min 95% CI':>24} {'Max Delta':>12} {'Max 95% CI':>24}")
print("-" * 105)
for s in strategy_names:
    p_mins = np.array(curb_loo_intervals[s]['payoff']['min'])
    p_maxs = np.array(curb_loo_intervals[s]['payoff']['max'])

    if abs(np.mean(p_mins)) < 0.0001 and abs(np.mean(p_maxs)) < 0.0001:
        continue

    p_min_mean = np.mean(p_mins)
    p_min_lo, p_min_hi = np.percentile(p_mins, [2.5, 97.5])
    p_max_mean = np.mean(p_maxs)
    p_max_lo, p_max_hi = np.percentile(p_maxs, [2.5, 97.5])

    print(f"  {s:<30} {p_min_mean:>+.4f} [{p_min_lo:>+.4f}, {p_min_hi:>+.4f}]   {p_max_mean:>+.4f} [{p_max_lo:>+.4f}, {p_max_hi:>+.4f}]")

    


Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
  100/100 done
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
  100/100 done
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
Greater than 20 going to curb
  100/100 done
Greater than 20 going to curb
Greater than 20 going to 

In [45]:
print("\n=== CURB-Conditional LOO Intervals ===")
print(f"{'Strategy':<30} {'Min Delta':>12} {'Min 95% CI':>24} {'Max Delta':>12} {'Max 95% CI':>24}")
print("-" * 105)
for s in strategy_names:
    p_mins = np.array(curb_loo_intervals[s]['payoff']['min'])
    p_maxs = np.array(curb_loo_intervals[s]['payoff']['max'])

    # if abs(np.mean(p_mins)) < 0.0001 and abs(np.mean(p_maxs)) < 0.0001:
    #     continue

    p_min_mean = np.mean(p_mins)
    p_min_lo, p_min_hi = np.percentile(p_mins, [2.5, 97.5])
    p_max_mean = np.mean(p_maxs)
    p_max_lo, p_max_hi = np.percentile(p_maxs, [2.5, 97.5])

    print(f"  {s:<30} {p_min_mean:>+.4f} [{p_min_lo:>+.4f}, {p_min_hi:>+.4f}]   {p_max_mean:>+.4f} [{p_max_lo:>+.4f}, {p_max_hi:>+.4f}]")


=== CURB-Conditional LOO Intervals ===
Strategy                          Min Delta               Min 95% CI    Max Delta               Max 95% CI
---------------------------------------------------------------------------------------------------------
  Tit For Tat                    -0.0000 [-0.0000, -0.0000]   +0.0000 [+0.0000, +0.0000]
  Tit For 2 Tats                 -0.0211 [-0.1454, -0.0000]   +0.7646 [+0.2263, +2.0925]
  Cooperator                     -0.0021 [-0.0051, +0.0034]   +2.0845 [+0.0316, +2.9398]
  General Soft Grudger: n=1,d=4,c=2 -0.0000 [-0.0000, -0.0000]   +0.0000 [+0.0000, +0.0000]
  Win-Stay Lose-Shift            -0.3916 [-0.4234, -0.3506]   -0.2662 [-0.4234, +1.3213]
  Grudger                        -0.0000 [-0.0000, -0.0000]   +0.0000 [+0.0000, +0.0000]
  Defector                       -0.0000 [-0.0000, -0.0000]   +0.0000 [+0.0000, +0.0000]
  Suspicious Tit For Tat         -0.0000 [-0.0000, -0.0000]   +0.0000 [+0.0000, +0.0000]
  Hard Go By Majority           

In [ ]:
#curbset analysis generates interval figures.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

metrics = ['payoff', 'coop']
metric_labels = {"payoff": 'Payoff', 'coop': "Cooperation"}

plot_agents = []
for s in strategy_names:
    p_mins = np.array(curb_loo_intervals[s]['payoff']['min'])
    p_maxs = np.array(curb_loo_intervals[s]['payoff']['max'])
    # if np.percentile(np.abs(p_mins), 95) < 0.001 and np.percentile(np.abs(p_maxs), 95) < 0.001:
    #       continue
    plot_agents.append(s)

#sort maxdelta of payoff 
l3_max = [np.mean(curb_loo_intervals[s]['payoff']['max']) for s in plot_agents]
order = np.argsort(l3_max)
plot_agents = [plot_agents[i] for i in order]
y = np.arange(len(plot_agents))

fig, axes = plt.subplots(1, len(metrics), 
                         figsize=(6 * len(metrics), 
                                  max(7, len(plot_agents) * 0.6)), 
                                  sharey=True)


for ax, m in zip(axes, metrics):
    for i, s in enumerate(plot_agents):
        p_mins = np.array(curb_loo_intervals[s][m]['min'])
        p_maxs = np.array(curb_loo_intervals[s][m]['max'])
        min_mean = np.mean(p_mins)
        min_lo, min_hi = np.percentile(p_mins, [2.5, 97.5])
        max_mean = np.mean(p_maxs)
        max_lo, max_hi = np.percentile(p_maxs, [2.5, 97.5])
        #min delta: red square + error bar
        ax.errorbar(min_mean, y[i] - 0.15,
              xerr=[[abs(min_mean - min_lo)], [abs(min_hi - min_mean)]],
              fmt='s', color='firebrick', markersize=5, capsize=3,
              elinewidth=1.5, zorder=3)

        ax.errorbar(max_mean, y[i] + 0.15,
                    xerr=[[abs(max_mean - max_lo)], [abs(max_hi - max_mean)]],
                    fmt='D', color='forestgreen', markersize=5, capsize=3,
                    elinewidth=1.5, zorder=3)

        #shaded region between means
        ax.fill_betweenx([y[i] - 0.08, y[i] + 0.08], min_mean, max_mean,
                        color='steelblue', alpha=0.15, zorder=1)
    ax.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(metric_labels[m], fontweight='bold')
    ax.grid(axis='x', alpha=0.2)
    ax.set_xlabel('Δ Metric')


axes[0].set_yticks(y)
axes[0].set_yticklabels(plot_agents, fontsize=9)

legend_elements = [
    Line2D([0], [0], marker='s', color='firebrick', label='Min Δ mean ± 95% CI',
            markersize=6, linestyle='None'),
    Line2D([0], [0], marker='D', color='forestgreen', label='Max Δ mean ± 95% CI',
            markersize=6, linestyle='None'),
]
axes[-1].legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.suptitle('CURB-Conditional LOO Intervals (Iterated PD)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pd_curb_loo_intervals.png', dpi=150, bbox_inches='tight')
plt.show()



/var/folders/fh/fwc37qhn04d8sxp65hwv1kxm0000gn/T/ipykernel_16513/2105989672.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
s_pred = 'Predator'
p_mins = np.array(curb_loo_intervals[s_pred]['payoff']['min'])
p_maxs = np.array(curb_loo_intervals[s_pred]['payoff']['max'])
print(f"Predator payoff min: mean={np.mean(p_mins):.4f}, CI=[{np.percentile(p_mins, 2.5):.4f}, {np.percentile(p_mins, 97.5):.4f}]")
print(f"Predator payoff max: mean={np.mean(p_maxs):.4f}, CI=[{np.percentile(p_maxs, 2.5):.4f}, {np.percentile(p_maxs, 97.5):.4f}]")
print(f"Non-zero mins: {np.sum(np.abs(p_mins) > 0.000000000001)}/{len(p_mins)}")
print(f"Non-zero maxs: {np.sum(np.abs(p_maxs) > 0.000000000001)}/{len(p_maxs)}")

SyntaxError: 'continue' not properly in loop (3666888298.py, line 5)

In [63]:
full_game = frozenset(range(n))
print(f"Full game in all_curbs: {full_game in set(frozenset(c) for c in all_curbs)}")

Full game in all_curbs: True


In [50]:
from collections import Counter

membership_count = Counter()
for c, count in curb_survival.items():
    if len(c) >= 2:
        for s_idx in c:
            membership_count[strategy_names[s_idx]] += count

print("Non-singleton CURB set appearances:")
for s, count in sorted(membership_count.items(), key=lambda x: -x[1])[:15]:
    print(f"  {s:<30} {count}")

print(f"\nStrategies never in non-singleton CURB sets:")
for s in strategy_names:
    if s not in membership_count or membership_count[s] == 0:
        print(f"  {s}")

Non-singleton CURB set appearances:
  Cooperator                     9193
  Tit For 2 Tats                 9041
  Soft Joss: 0.9                 8331
  General Soft Grudger: n=1,d=4,c=2 8103
  Evolved FSM 4                  8073
  Adaptive Tit For Tat: 0.5      7903
  Prober 2                       7812
  Hard Go By Majority            6961
  ThueMorse                      6708
  Detective                      6621
  Win-Stay Lose-Shift            6306
  Tit For Tat                    6229
  DoubleCrosser: (D, D)          6217
  Bully                          6073
  Calculator                     5574

Strategies never in non-singleton CURB sets:
